# Kolmogorov-Arnold Networks (KANs)

---
### References
1. Liu, Z. et al. (2024) 'KAN: Kolmogorov-Arnold Networks', *arXiv*. https://arxiv.org/abs/2404.19756
2. Kolmogorov, A.N. (1957) 'On the representation of continuous functions of many variables', *Doklady Akademii Nauk SSSR*, 114(5).
3. Arnold, V.I. (1963) 'On functions of three variables', *Doklady Akademii Nauk SSSR*, 114(5).
4. Hornik, K. (1991) 'Approximation capabilities of multilayer feedforward networks', *Neural Networks*, 4(2). https://doi.org/10.1016/0893-6080(91)90009-T
5. Liu, Z. et al. (2024) 'KAN 2.0: Kolmogorov-Arnold Networks Meet Science', *arXiv*. https://arxiv.org/abs/2408.10205

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.patches import FancyBboxPatch
import warnings, os
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

os.makedirs('figures_kan', exist_ok=True)
torch.manual_seed(42); np.random.seed(42)

BG='#0F1117'; PANEL='#1A1D2E'; BORDER='#2D3154'
TEXT='#E8EAF6'; SUBTEXT='#8892B0'; GRID='#1E2140'
C1='#4F9CF9'; C2='#FF6B6B'; C3='#43D9AD'; C4='#FFB86C'
C5='#BD93F9'; C6='#FF79C6'; C7='#50FA7B'; C8='#F1FA8C'

plt.rcParams.update({
    'figure.facecolor':BG,'axes.facecolor':PANEL,'axes.edgecolor':BORDER,
    'axes.labelcolor':SUBTEXT,'axes.titlecolor':TEXT,'xtick.color':SUBTEXT,
    'ytick.color':SUBTEXT,'text.color':TEXT,'grid.color':GRID,
    'grid.linewidth':0.6,'font.family':'DejaVu Sans',
    'axes.spines.top':False,'axes.spines.right':False,
    'legend.facecolor':PANEL,'legend.edgecolor':BORDER,
})
def sax(ax): ax.set_facecolor(PANEL); ax.grid(True,alpha=0.15); [sp.set_edgecolor(BORDER) for sp in ax.spines.values()]
print('Setup complete.')

Setup complete.


## Figure 1 MLP vs KAN Architecture

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 7.5), facecolor=BG)
fig.patch.set_facecolor(BG)
for ax in axes:
    ax.set_facecolor(BG); ax.axis('off')
    ax.set_xlim(0, 10); ax.set_ylim(0, 10)

def draw_node(ax, x, y, r, col, label='', alpha=1.0):
    c = plt.Circle((x,y), r, color=col, alpha=alpha, zorder=4)
    g = plt.Circle((x,y), r*1.6, color=col, alpha=alpha*0.12, zorder=3)
    ax.add_patch(c); ax.add_patch(g)
    if label: ax.text(x, y, label, ha='center', va='center', fontsize=8, color='white', fontweight='bold', zorder=5)

def draw_spline_edge(ax, x1, y1, x2, y2, col, activated=True):
    """Draw an edge with a small spline visualisation on it."""
    mx, my = (x1+x2)/2, (y1+y2)/2
    ax.plot([x1,x2],[y1,y2], color=col, lw=0.8, alpha=0.3, zorder=2)
    if activated:
        # Draw tiny spline curve at midpoint
        t = np.linspace(-0.4, 0.4, 30)
        spline_y = 0.2 * np.sin(3*t) * np.exp(-t**2)
        # Rotate to align with edge direction
        angle = np.arctan2(y2-y1, x2-x1)
        rot_x = mx + t*np.cos(angle) - spline_y*np.sin(angle)
        rot_y = my + t*np.sin(angle) + spline_y*np.cos(angle)
        ax.plot(rot_x, rot_y, color=col, lw=2.0, alpha=0.85, zorder=3,
                path_effects=[pe.withStroke(linewidth=3.5, foreground=col+'44')])

# LEFT: MLP
ax = axes[0]
ax.text(5, 9.6, 'MLP  Fixed activations on NODES', ha='center', fontsize=12, fontweight='bold', color=C1)
ax.text(5, 9.1, 'σ(Wx + b)  — σ is fixed (ReLU, tanh, sigmoid)', ha='center', fontsize=9.5, color=SUBTEXT)

layers_mlp = [
    [(3.0, 7.5), (3.0, 6.0), (3.0, 4.5)],   # input
    [(5.5, 7.8), (5.5, 6.5), (5.5, 5.2), (5.5, 3.9)],  # hidden
    [(8.0, 6.5), (8.0, 5.0)],  # output
]
layer_cols = [C1, C3, C2]
layer_labels = [['x₁','x₂','x₃'], ['h₁','h₂','h₃','h₄'], ['y₁','y₂']]
layer_names = ['Input\nlayer', 'Hidden\nlayer\n(fixed σ)', 'Output\nlayer']

for li, (layer, col, lbs) in enumerate(zip(layers_mlp, layer_cols, layer_labels)):
    for (x,y), lb in zip(layer, lbs):
        draw_node(ax, x, y, 0.3, col, lb)
    ys = [p[1] for p in layer]
    ax.text(layer[0][0], max(ys)+0.7, layer_names[li], ha='center', fontsize=8.5, color=col)

# Edges with fixed activation symbol
for (x1,y1) in layers_mlp[0]:
    for (x2,y2) in layers_mlp[1]:
        ax.plot([x1+0.3,x2-0.3],[y1,y2], color=C1, lw=0.7, alpha=0.35, zorder=2)
for (x1,y1) in layers_mlp[1]:
    for (x2,y2) in layers_mlp[2]:
        ax.plot([x1+0.3,x2-0.3],[y1,y2], color=C3, lw=0.7, alpha=0.35, zorder=2)

# Fixed activation box on hidden nodes
for (x,y) in layers_mlp[1]:
    ax.text(x, y-0.55, 'σ', ha='center', fontsize=9, color=C4, fontweight='bold')

ax.text(5, 2.8, 'Weights: learnable (W, b)', ha='center', fontsize=9.5, color=SUBTEXT)
ax.text(5, 2.3, 'Activations: FIXED (same σ everywhere)', ha='center', fontsize=9.5, color=C2)
r=FancyBboxPatch((1,1.6),8,0.9,boxstyle='round,pad=0.1',facecolor=C2,edgecolor=C2,alpha=0.15,zorder=3)
ax.add_patch(r)
ax.text(5, 2.05, '⚠  Fixed activation functions limited expressiveness per edge', ha='center', fontsize=9, color=C2, zorder=4, fontweight='bold')

#RIGHT: KAN
ax = axes[1]
ax.text(5, 9.6, 'KAN Learnable activations on EDGES', ha='center', fontsize=12, fontweight='bold', color=C3)
ax.text(5, 9.1, 'Σ φᵢⱼ(xᵢ)  - each φᵢⱼ is a learnable spline', ha='center', fontsize=9.5, color=SUBTEXT)

layers_kan = [
    [(2.5, 7.5), (2.5, 5.5), (2.5, 3.5)],
    [(5.5, 7.5), (5.5, 5.5), (5.5, 3.5)],
    [(8.5, 6.5), (8.5, 4.5)],
]
for li, (layer, col) in enumerate(zip(layers_kan, layer_cols)):
    for x,y in layer:
        draw_node(ax, x, y, 0.28, col)

# Edges with learnable spline curves
spline_cols = [[C3,C5,C4,C6,C7,C8,C1,C2,C3][:len(layers_kan[0])*len(layers_kan[1])],
               [C3,C4,C5,C1,C2,C6][:len(layers_kan[1])*len(layers_kan[2])]]
ci = 0
for (x1,y1) in layers_kan[0]:
    for (x2,y2) in layers_kan[1]:
        draw_spline_edge(ax, x1+0.28, y1, x2-0.28, y2,
                         EXPERT_COLS[ci%8] if 'EXPERT_COLS' in dir() else [C3,C5,C4,C6,C7,C1,C2,C8][ci%8])
        ci += 1
ci = 0
for (x1,y1) in layers_kan[1]:
    for (x2,y2) in layers_kan[2]:
        draw_spline_edge(ax, x1+0.28, y1, x2-0.28, y2, [C3,C4,C1,C5,C2,C6][ci%6])
        ci += 1

ax.text(5, 2.6, 'Each edge has its own learnable φ(x) — a B-spline', ha='center', fontsize=9.5, color=SUBTEXT)
ax.text(5, 2.1, 'Activations: LEARNABLE (different shape per edge)', ha='center', fontsize=9.5, color=C3)
r=FancyBboxPatch((1,1.4),8,0.9,boxstyle='round,pad=0.1',facecolor=C3,edgecolor=C3,alpha=0.15,zorder=3)
ax.add_patch(r)
ax.text(5, 1.85, '✓  Every edge learns its own activation shape', ha='center', fontsize=9, color=C3, zorder=4, fontweight='bold')

fig.text(0.5, 1.01, 'MLP vs KAN — Where Activations Live',
         ha='center', fontsize=15, fontweight='bold', color=TEXT)
fig.text(0.5, 0.975,
         'MLP: fixed activation on nodes | KAN: learnable spline activation on every edge',
         ha='center', fontsize=10.5, color=SUBTEXT)
plt.tight_layout()
plt.savefig('figures_kan/fig1_mlp_vs_kan.png', dpi=180, bbox_inches='tight', facecolor=BG)
plt.show(); print('Figure 1 saved.')

Figure 1 saved.


## KAN Implementation + Figure 2  Learnable Splines

In [ ]:
class BSplineBasis(nn.Module):
    """B-spline basis functions for KAN activation."""
    def __init__(self, grid_size=5, spline_order=3, x_range=(-1,1)):
        super().__init__()
        self.grid_size   = grid_size
        self.spline_order= spline_order
        h = (x_range[1]-x_range[0]) / grid_size
        grid = torch.arange(-spline_order, grid_size+spline_order+1) * h + x_range[0]
        self.register_buffer('grid', grid)

    def forward(self, x):
        x = x.unsqueeze(-1)
        bases = ((x >= self.grid[:-1]) & (x < self.grid[1:])).float()
        for k in range(1, self.spline_order+1):
            bases = (
                (x - self.grid[:-(k+1)]) / (self.grid[k:-1] - self.grid[:-(k+1)]) * bases[..., :-1] +
                (self.grid[k+1:] - x)    / (self.grid[k+1:] - self.grid[1:-k])   * bases[..., 1:]
            )
        return bases


class KANLayer(nn.Module):
    """Single KAN layer: output_j = Σ_i φ_ij(x_i)."""
    def __init__(self, in_dim, out_dim, grid_size=5, spline_order=3):
        super().__init__()
        self.in_dim  = in_dim
        self.out_dim = out_dim
        self.basis   = BSplineBasis(grid_size, spline_order)
        n_basis = grid_size + spline_order
        # Learnable spline coefficients one set per (in, out) pair
        self.coeff = nn.Parameter(torch.randn(out_dim, in_dim, n_basis) * 0.1)
        # Residual linear connection (SiLU base)
        self.scale = nn.Parameter(torch.ones(out_dim, in_dim))

    def forward(self, x):
        # x: (B, in_dim)
        B = x.shape[0]
        basis = self.basis(x)  # (B, in_dim, n_basis)
        # Spline activation for each (in, out) pair
        spline = torch.einsum('bin,oin->bo', basis, self.coeff)  # (B, out_dim)
        # Base activation (SiLU / swish)
        base = torch.einsum('bi,oi->bo', F.silu(x), self.scale)  # (B, out_dim)
        return spline + base


class KAN(nn.Module):
    """Multi-layer KAN."""
    def __init__(self, dims, grid_size=5, spline_order=3):
        super().__init__()
        self.layers = nn.ModuleList([
            KANLayer(dims[i], dims[i+1], grid_size, spline_order)
            for i in range(len(dims)-1)
        ])
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

print('KAN classes defined.')

KAN classes defined.


In [ ]:
#  Visualise learned spline activations
# Train a simple KAN on a 1D function approximation
x_1d = torch.linspace(-1, 1, 200).unsqueeze(1)
y_target = torch.sin(3*x_1d) + 0.5*torch.cos(7*x_1d)  # complex target

kan_1d = KAN([1, 8, 1], grid_size=8, spline_order=3)
opt = torch.optim.Adam(kan_1d.parameters(), lr=5e-3)
for ep in range(3000):
    opt.zero_grad()
    loss = F.mse_loss(kan_1d(x_1d), y_target)
    loss.backward(); opt.step()

with torch.no_grad():
    y_pred = kan_1d(x_1d).squeeze().numpy()

# Extract learned spline shapes from layer 1
fig, axes = plt.subplots(2, 4, figsize=(18, 8), gridspec_kw={'hspace':0.45,'wspace':0.25})
fig.patch.set_facecolor(BG)

x_vis = torch.linspace(-1, 1, 100).unsqueeze(1)
kan_1d.eval()
layer = kan_1d.layers[0]
basis_vis = layer.basis(x_vis.squeeze())  # (100, n_basis)
x_np = x_vis.squeeze().numpy()

ECOLS = [C1,C2,C3,C4,C5,C6,C7,C8]
for idx in range(8):
    ax = axes[idx//4][idx%4]
    sax(ax)
    with torch.no_grad():
        spline_activation = (basis_vis @ layer.coeff[idx, 0]).numpy()  # learned φ_{idx,0}
        base_activation   = (F.silu(x_vis.squeeze()) * layer.scale[idx, 0]).numpy()
        total = spline_activation + base_activation
    ax.fill_between(x_np, total, alpha=0.2, color=ECOLS[idx])
    ax.plot(x_np, spline_activation, '--', color=SUBTEXT, lw=1.5, alpha=0.7, label='Spline φ')
    ax.plot(x_np, base_activation,   ':', color=C4, lw=1.5, alpha=0.7, label='Base (SiLU)')
    ax.plot(x_np, total, color=ECOLS[idx], lw=2.5,
            path_effects=[pe.withStroke(linewidth=4, foreground=ECOLS[idx]+'44')],
            label='Total φ_{i,j}')
    ax.set_title(f'Edge activation φ_{{0,{idx+1}}}', fontsize=10, fontweight='bold', color=ECOLS[idx])
    ax.set_xlabel('x', fontsize=9, color=SUBTEXT)
    if idx == 0: ax.legend(fontsize=7, loc='upper right')

fig.text(0.5, 1.01, 'Learnable Spline Activations — Each Edge Has Its Own Shape',
         ha='center', fontsize=15, fontweight='bold', color=TEXT)
fig.text(0.5, 0.975,
         'Each of the 8 edges from input to hidden layer has learned a different activation function (B-spline + SiLU base)',
         ha='center', fontsize=10.5, color=SUBTEXT)
plt.savefig('figures_kan/fig2_learned_splines.png', dpi=180, bbox_inches='tight', facecolor=BG)
plt.show(); print('Figure 2 saved.')

Figure 2 saved.


## Figure 3 KAN vs MLP Function Approximation

In [ ]:
# Compare KAN vs MLP on function approximation tasks
functions = [
    ('sin(3x) + 0.5cos(7x)', lambda x: np.sin(3*x) + 0.5*np.cos(7*x)),
    ('x² · exp(−x²)', lambda x: x**2 * np.exp(-x**2)),
    ('|sin(πx)|', lambda x: np.abs(np.sin(np.pi*x))),
]

x_train = np.linspace(-1, 1, 150).astype(np.float32)
x_plot  = np.linspace(-1.3, 1.3, 300).astype(np.float32)  # slightly OOD

fig, axes = plt.subplots(len(functions), 3, figsize=(18, 10),
                          gridspec_kw={'hspace':0.5,'wspace':0.1})
fig.patch.set_facecolor(BG)

for row, (fname, fn) in enumerate(functions):
    y_train = fn(x_train) + np.random.randn(len(x_train))*0.05
    y_true  = fn(x_plot)

    X_t = torch.FloatTensor(x_train).unsqueeze(1)
    Y_t = torch.FloatTensor(y_train).unsqueeze(1)
    X_p = torch.FloatTensor(x_plot).unsqueeze(1)

    # MLP
    mlp = nn.Sequential(nn.Linear(1,32),nn.Tanh(),nn.Linear(32,32),nn.Tanh(),nn.Linear(32,1))
    opt_m = torch.optim.Adam(mlp.parameters(), lr=5e-3)
    for _ in range(2000):
        opt_m.zero_grad(); F.mse_loss(mlp(X_t),Y_t).backward(); opt_m.step()

    # KAN
    kan = KAN([1,8,1], grid_size=8, spline_order=3)
    opt_k = torch.optim.Adam(kan.parameters(), lr=5e-3)
    for _ in range(2000):
        opt_k.zero_grad(); F.mse_loss(kan(X_t),Y_t).backward(); opt_k.step()

    mlp.eval(); kan.eval()
    with torch.no_grad():
        mlp_pred = mlp(X_p).squeeze().numpy()
        kan_pred = kan(X_p).squeeze().numpy()

    mlp_mse = np.mean((mlp_pred[50:250] - y_true[50:250])**2)
    kan_mse = np.mean((kan_pred[50:250] - y_true[50:250])**2)

    for col, (model_pred, model_name, model_col, mse) in enumerate([
        (None, 'True function', SUBTEXT, None),
        (mlp_pred, 'MLP (32-32)', C1, mlp_mse),
        (kan_pred, 'KAN [1,8,1]', C3, kan_mse),
    ]):
        ax = axes[row][col]
        sax(ax)
        ax.axvspan(-1.3,-1, alpha=0.07, color=C2, label='OOD')
        ax.axvspan(1, 1.3,  alpha=0.07, color=C2)
        ax.plot(x_plot, y_true, '--', color=SUBTEXT, lw=1.8, alpha=0.7, label='True')
        ax.scatter(x_train[::5], y_train[::5], c=TEXT, s=15, alpha=0.5, zorder=4)
        if model_pred is not None:
            ax.plot(x_plot, model_pred, color=model_col, lw=2.5,
                    path_effects=[pe.withStroke(linewidth=5,foreground=model_col+'44')])
            ax.set_title(f'{model_name}\nMSE={mse:.4f}', fontsize=10, fontweight='bold', color=model_col, pad=6)
        else:
            ax.set_title(f'f(x) = {fname}', fontsize=10, fontweight='bold', color=TEXT, pad=6)
        ax.set_xlim(-1.3, 1.3)
        ax.set_xlabel('x', color=SUBTEXT, fontsize=9)
        if col == 0: ax.set_ylabel(f'f(x)', color=SUBTEXT, fontsize=9)

fig.text(0.5, 1.01, 'KAN vs MLP Function Approximation on Three Tasks',
         ha='center', fontsize=15, fontweight='bold', color=TEXT)
fig.text(0.5, 0.975,
         'KAN achieves better MSE on smooth functions; red shading = OOD region (outside training range)',
         ha='center', fontsize=10.5, color=SUBTEXT)
plt.savefig('figures_kan/fig3_kan_vs_mlp.png', dpi=180, bbox_inches='tight', facecolor=BG)
plt.show(); print('Figure 3 saved.')

Figure 3 saved.


## Figure 4 Kolmogorov-Arnold Theorem Visualised

In [ ]:
fig = plt.figure(figsize=(16, 6.5), facecolor=BG)
gs = GridSpec(1, 3, figure=fig, wspace=0.2)
fig.patch.set_facecolor(BG)

# Panel 1: Kolmogorov-Arnold theorem statement
ax1 = fig.add_subplot(gs[0])
ax1.set_facecolor(PANEL); ax1.axis('off')
ax1.set_xlim(0,10); ax1.set_ylim(0,10)

ax1.text(5, 9.5, 'Kolmogorov-Arnold\nRepresentation Theorem', ha='center', fontsize=11,
         fontweight='bold', color=C3, multialignment='center')

# Theorem box
r=FancyBboxPatch((0.3,5.0),9.4,4.0,boxstyle='round,pad=0.15',facecolor=C3,edgecolor=C3,alpha=0.12,zorder=3)
ax1.add_patch(r)
thm_lines = [
    (5, 8.5, 'Any continuous function', 10.5, TEXT),
    (5, 8.0, 'f: [0,1]ⁿ → ℝ', 12, C3),
    (5, 7.3, 'can be represented as:', 10.5, TEXT),
    (5, 6.7, 'f(x₁,...,xₙ) =', 11, C4),
    (5, 6.1, 'Σ_{q=0}^{2n} Φ_q( Σ_{p=1}^{n} φ_{q,p}(x_p) )', 10, C3),
    (5, 5.5, 'where Φ_q, φ_{q,p} are', 9.5, SUBTEXT),
    (5, 5.1, 'univariate functions', 9.5, SUBTEXT),
]
for x,y,t,fs,col in thm_lines:
    ax1.text(x, y, t, ha='center', va='center', fontsize=fs, color=col, fontweight='bold' if col==C3 else 'normal', zorder=4)

# Implication
r2=FancyBboxPatch((0.3,2.5),9.4,2.2,boxstyle='round,pad=0.15',facecolor=C4,edgecolor=C4,alpha=0.18,zorder=3)
ax1.add_patch(r2)
ax1.text(5, 3.95, '→ KAN insight:', ha='center', fontsize=10, color=C4, fontweight='bold', zorder=4)
ax1.text(5, 3.45, 'Multivariate functions can be', ha='center', fontsize=9.5, color=TEXT, zorder=4)
ax1.text(5, 3.0, 'decomposed into 1D functions!', ha='center', fontsize=9.5, color=TEXT, zorder=4)
ax1.text(5, 2.55, 'Learn those 1D functions on edges.', ha='center', fontsize=9, color=SUBTEXT, zorder=4)

ax1.text(5, 1.5, 'Published: Kolmogorov (1957)', ha='center', fontsize=9, color=SUBTEXT)
ax1.text(5, 1.0, 'Applied to NNs: Liu et al. (2024)', ha='center', fontsize=9, color=C3)

# Panel 2: B-spline basis functions
ax2 = fig.add_subplot(gs[1])
sax(ax2)
x_s = np.linspace(-1, 1, 300)
# Show multiple B-spline basis functions
grid = np.linspace(-1, 1, 6)
colors_s = [C1, C2, C3, C4, C5, C6, C7]
for i in range(5):
    # Cubic B-spline basis approximation
    center = (grid[i] + grid[i+1]) / 2
    width  = (grid[1] - grid[0]) * 1.8
    basis  = np.exp(-((x_s - center)/width)**2 * 4)
    # Zero outside support
    support = (x_s >= grid[i]-width) & (x_s <= grid[i+1]+width)
    y_b = np.where(support, basis, 0)
    ax2.fill_between(x_s, y_b, alpha=0.25, color=colors_s[i])
    ax2.plot(x_s, y_b, color=colors_s[i], lw=2.0,
             path_effects=[pe.withStroke(linewidth=3,foreground=colors_s[i]+'44')],
             label=f'B_{i}(x)')

ax2.set_xlabel('x', color=SUBTEXT, fontsize=11)
ax2.set_ylabel('Basis value', color=SUBTEXT, fontsize=11)
ax2.set_title('B-Spline Basis Functions\nLinear combination = any smooth curve',
              fontsize=10, fontweight='bold', color=TEXT, pad=8)
ax2.legend(fontsize=8, ncol=2)

# Panel 3: Learnable combination
ax3 = fig.add_subplot(gs[2])
sax(ax3)
x_s2 = np.linspace(-1, 1, 300)
# Show how different coefficient combinations produce different functions
np.random.seed(42)
for trial in range(6):
    coeffs = np.random.randn(5) * 0.8
    y_combo = np.zeros_like(x_s2)
    for i in range(5):
        center = (grid[i]+grid[i+1])/2
        width  = (grid[1]-grid[0])*1.8
        basis  = np.exp(-((x_s2-center)/width)**2*4)
        support=(x_s2>=grid[i]-width)&(x_s2<=grid[i+1]+width)
        y_combo += coeffs[i]*np.where(support,basis,0)
    ax3.plot(x_s2, y_combo, color=colors_s[trial], lw=2.0, alpha=0.85,
             path_effects=[pe.withStroke(linewidth=3,foreground=colors_s[trial]+'44')],
             label=f'c={np.round(coeffs[:2],1)}')

ax3.set_xlabel('x', color=SUBTEXT, fontsize=11)
ax3.set_ylabel('φ(x)', color=SUBTEXT, fontsize=11)
ax3.set_title('Learnable Spline φ(x) = Σ cᵢBᵢ(x)\nDifferent coefficients → different shapes',
              fontsize=10, fontweight='bold', color=TEXT, pad=8)
ax3.legend(fontsize=7, loc='upper right')

fig.text(0.5, 1.01, 'The Mathematical Foundation of KANs',
         ha='center', fontsize=15, fontweight='bold', color=TEXT)
fig.text(0.5, 0.975,
         'Kolmogorov-Arnold theorem (1957) → decompose multivariate functions into univariate B-splines on edges',
         ha='center', fontsize=10.5, color=SUBTEXT)
plt.savefig('figures_kan/fig4_theory.png', dpi=180, bbox_inches='tight', facecolor=BG)
plt.show(); print('Figure 4 saved.')

Figure 4 saved.


## Figure 5 KAN Interpretability and Symbolic Regression

In [ ]:
# KAN on real classification + interpretability demo
X, y = make_classification(n_samples=1000, n_features=4, n_informative=3,
                            n_redundant=1, random_state=42)
scaler = StandardScaler(); X = scaler.fit_transform(X).astype(np.float32)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
X_tr_t=torch.FloatTensor(X_tr); y_tr_t=torch.LongTensor(y_tr)
X_te_t=torch.FloatTensor(X_te); y_te_t=torch.LongTensor(y_te)

# Train KAN and MLP
kan_cls = KAN([4, 8, 4, 2], grid_size=5, spline_order=3)
mlp_cls = nn.Sequential(nn.Linear(4,32),nn.ReLU(),nn.Linear(32,16),nn.ReLU(),nn.Linear(16,2))

for model, name in [(kan_cls,'KAN'), (mlp_cls,'MLP')]:
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    for _ in range(1500):
        opt.zero_grad()
        F.cross_entropy(model(X_tr_t), y_tr_t).backward(); opt.step()
    model.eval()
    with torch.no_grad():
        acc = (model(X_te_t).argmax(1)==y_te_t).float().mean()
    print(f'{name} accuracy: {acc*100:.1f}%')

# Interpretability: visualise what each edge learned
fig, axes = plt.subplots(2, 4, figsize=(18, 8), gridspec_kw={'hspace':0.5,'wspace':0.25})
fig.patch.set_facecolor(BG)

x_range = torch.linspace(-3, 3, 100).unsqueeze(1)
x_4d = torch.zeros(100, 4)
feat_names = ['Feature 1', 'Feature 2', 'Feature 3', 'Feature 4']

kan_cls.eval()
layer0 = kan_cls.layers[0]
basis_v = layer0.basis(x_range.squeeze())  # (100, n_basis)
x_np_4 = x_range.squeeze().numpy()

for feat_idx in range(4):
    for row in range(2):
        ax = axes[row][feat_idx]
        sax(ax)
        # Compute activation for this feature → output node
        out_node = row
        with torch.no_grad():
            spline_act = (basis_v @ layer0.coeff[out_node, feat_idx]).numpy()
            base_act   = (F.silu(x_range.squeeze()) * layer0.scale[out_node, feat_idx]).numpy()
            total_act  = spline_act + base_act
        col = [C1,C2,C3,C4][feat_idx]
        ax.axhline(0, color=SUBTEXT, lw=0.8, alpha=0.4)
        ax.fill_between(x_np_4, total_act, 0, alpha=0.2, color=col)
        ax.plot(x_np_4, total_act, color=col, lw=2.5,
                path_effects=[pe.withStroke(linewidth=4,foreground=col+'44')])
        ax.set_title(f'φ({feat_names[feat_idx]} → Out{out_node+1})',
                     fontsize=9.5, fontweight='bold', color=col, pad=5)
        ax.set_xlabel('Input value', fontsize=8, color=SUBTEXT)
        ax.set_ylabel('Activation', fontsize=8, color=SUBTEXT)

fig.text(0.5, 1.01, 'KAN Interpretability Each Edge Is a Readable Function',
         ha='center', fontsize=15, fontweight='bold', color=TEXT)
fig.text(0.5, 0.975,
         'Unlike MLP weights, KAN edge functions can be visualised and potentially identified as known mathematical functions',
         ha='center', fontsize=10.5, color=SUBTEXT)
plt.savefig('figures_kan/fig5_interpretability.png', dpi=180, bbox_inches='tight', facecolor=BG)
plt.show(); print('Figure 5 saved.')

KAN accuracy: 90.5%


MLP accuracy: 95.5%


Figure 5 saved.


## Summary

| Property | MLP | KAN |
|----------|-----|-----|
| Activations | Fixed σ on nodes | Learnable φ on edges |
| Parameters | Weights + biases | Spline coefficients |
| Foundation | Universal approximation | Kolmogorov-Arnold theorem |
| Interpretability | Low (weight matrices) | Higher (visualise edge functions) |
| Smooth functions | Good | Better (splines excel) |
| Year | 1986+ | 2024 |